# Handling message blocks

With tools, Claude's reply is no longer one text block. `response.content` is a **list of blocks**:

| Block | What it holds |
|---|---|
| `text` | what Claude says, e.g. "Let me check the time" (may be missing) |
| `tool_use` | a call request: `id` (to match the result later), `name`, `input` (arguments as a dict) |

The API is stateless, so the history must keep the **whole** assistant reply, every block,
not just the text: the next request has to show Claude its own `tool_use` block.

Full flow, this lesson does steps 1-2:

1. Send the user message and the tool schemas
2. Get an assistant message with text and `tool_use` blocks
3. Run the function with the requested arguments
4. Send the result back with the full history
5. Get the final answer

## Setup

In [1]:
from datetime import datetime

from anthropic import Anthropic
from anthropic.types import Message, ToolParam
from dotenv import load_dotenv

load_dotenv()
client = Anthropic()
model = "claude-haiku-4-5"


def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)


get_current_datetime_schema = ToolParam({
    "name": "get_current_datetime",
    "description": (
        "Returns the current date and time formatted according to the specified format. "
        "Use it whenever the answer depends on the current date or time, for example to "
        "compute a date relative to today. "
        "Returns the formatted date and time as a string."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "date_format": {
                "type": "string",
                "description": (
                    "A string specifying the format of the returned datetime. "
                    "Uses Python's strftime format codes, e.g. '%Y-%m-%d' or '%H:%M'."
                ),
                "default": "%Y-%m-%d %H:%M:%S",
            }
        },
        "required": [],
    },
})

## A request with tools

The code from the lesson: `tools` takes a list of schemas.

In [2]:
messages = []
messages.append({
    "role": "user",
    "content": "What is the exact time, formatted as HH:MM:SS?",
})

response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema],
)

print("stop_reason:", response.stop_reason)
for block in response.content:
    if block.type == "text":
        print(f"text block:     {block.text!r}")
    elif block.type == "tool_use":
        print(f"tool_use block: id={block.id} name={block.name} input={block.input}")

stop_reason: tool_use
tool_use block: id=toolu_bdrk_013kuiJ8JQwQdgmAijbAggsr name=get_current_datetime input={'date_format': '%H:%M:%S'}


## Keep every block in the history

`response.content` goes into the history as is, a list of blocks, not a string.

In [3]:
messages.append({
    "role": "assistant",
    "content": response.content,
})

for message in messages:
    print(message["role"], "->", message["content"])

user -> What is the exact time, formatted as HH:MM:SS?
assistant -> [ToolUseBlock(id='toolu_bdrk_013kuiJ8JQwQdgmAijbAggsr', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use', toolset_name=None)]


## Updated helpers

The old helpers took a string and `chat()` returned only text. With tools they need to:

- `add_user_message` / `add_assistant_message`: accept a string, a list of blocks or a whole `Message`
- `chat`: take `tools` and return the whole `Message`, so the caller sees `stop_reason` and every block
- `text_from_message`: get the text out when it is needed

The lesson only mentions this; the code is the course notebook's version.

In [4]:
def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=None, tools=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
    }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join(block.text for block in message.content if block.type == "text")

In [5]:
messages = []
add_user_message(messages, "What is today's date? Tell me what you are going to do first.")
response = chat(messages, tools=[get_current_datetime_schema])
add_assistant_message(messages, response)

print("stop_reason:", response.stop_reason)
print("text:", text_from_message(response))
print("blocks:", [block.type for block in messages[-1]["content"]])

stop_reason: tool_use
text: I'll first get the current date by calling the get_current_datetime function.
blocks: ['text', 'tool_use']


The conversation now ends with a `tool_use` request that nobody has answered.
Next lesson: run the function and send a `tool_result` back.